Francisco Beenita, 2026

# Computing Average Neighborhood Size

In [1]:
import os
import urllib.request
import numpy as np
import geopandas as gpd
import scipy.sparse as sparse
from scipy.spatial import cKDTree
import gc

# ========================================================
# 1. SETUP URLs
# ========================================================
data_urls = {
    'zacatecas': 'https://www.dropbox.com/scl/fi/mtw6yanftigtyafoc493l/zacatecas_guadalupe_final_results.gpkg?rlkey=uibkahfz9i55z4m5idn3r84ou&dl=1',
    'monterrey': 'https://www.dropbox.com/scl/fi/hss7xxnfrl8fwk4prupvr/monterrey_final_results.gpkg?rlkey=6kri4ejs5juhpm9l699nkg5j1&dl=1',
    'cdmx': 'https://www.dropbox.com/scl/fi/wxps4aijd4mf2jmji0j0n/ciudad_de_mexico_final_results.gpkg?rlkey=8k7904o43fe7d66v4vcgdeya2&dl=1'
}

# ========================================================
# 2. HELPER FUNCTIONS (No GAM, No Optimization)
# ========================================================
def download_and_filter(city, url):
    fname = f"{city}.gpkg"
    if not os.path.exists(fname):
        print(f"Downloading {city}...")
        urllib.request.urlretrieve(url, fname)
    gdf = gpd.read_file(fname)
    return gdf[(gdf['population_count_1_2km'] > 0) & (gdf['network_density_1_2km'].notnull())].copy().reset_index(drop=True)

def build_matrix(gdf, threshold=1200):
    print(f"Building KDTree & Matrix...")
    coords = np.column_stack((gdf.geometry.centroid.x, gdf.geometry.centroid.y))
    tree = cKDTree(coords)
    A_mats = []
    for i in range(0, coords.shape[0], 10000):
        end = min(i + 10000, coords.shape[0])
        dist_mat = cKDTree(coords[i:end]).sparse_distance_matrix(tree, max_distance=threshold).tocoo()
        dist_mat.data = (1.0 - (dist_mat.data / threshold)).astype(np.float32)
        A_mats.append(dist_mat.tocsr())
        del dist_mat; gc.collect()
    A = sparse.vstack(A_mats)
    A.setdiag(1.0)
    return A.tocsc()

# ========================================================
# 3. FAST CALCULATION EXECUTION
# ========================================================
print("--- Computing Average Neighborhood Densities (δ = 1200m) ---")
for city in ['zacatecas', 'monterrey', 'cdmx']:
    print(f"\nProcessing {city.capitalize()}...")
    gdf = download_and_filter(city, data_urls[city])
    A_matrix = build_matrix(gdf, threshold=1200)

    total_nodes = A_matrix.shape[0]
    total_connections = A_matrix.nnz
    avg_neighbors = total_connections / total_nodes

    print(f"Total Nodes (N): {total_nodes:,}")
    print(f"Total Graph Connections: {total_connections:,}")
    print(f"Average Neighborhood Size: {avg_neighbors:.1f} cells")

    # Free memory before the next city
    del gdf, A_matrix
    gc.collect()

--- Computing Average Neighborhood Densities (δ = 1200m) ---

Processing Zacatecas...
Building KDTree & Matrix...
Total Nodes (N): 42,782
Total Graph Connections: 16,485,178
Average Neighborhood Size: 385.3 cells

Processing Monterrey...
Building KDTree & Matrix...
Total Nodes (N): 303,782
Total Graph Connections: 148,853,230
Average Neighborhood Size: 490.0 cells

Processing Cdmx...
Building KDTree & Matrix...
Total Nodes (N): 539,179
Total Graph Connections: 316,720,827
Average Neighborhood Size: 587.4 cells


In [3]:
!pip install pygam

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.6/84.6 kB 6.2 MB/s eta 0:00:00


In [4]:
import os
import time
import urllib.request
import numpy as np
import geopandas as gpd
import scipy.sparse as sparse
from scipy.spatial import cKDTree
from pygam import LinearGAM, s
import heapq
import gc

# ========================================================
# 1. SETUP & DATA INGESTION
# ========================================================
data_urls = {
    'zacatecas': 'https://www.dropbox.com/scl/fi/mtw6yanftigtyafoc493l/zacatecas_guadalupe_final_results.gpkg?rlkey=uibkahfz9i55z4m5idn3r84ou&dl=1',
    'monterrey': 'https://www.dropbox.com/scl/fi/hss7xxnfrl8fwk4prupvr/monterrey_final_results.gpkg?rlkey=6kri4ejs5juhpm9l699nkg5j1&dl=1',
    'cdmx': 'https://www.dropbox.com/scl/fi/wxps4aijd4mf2jmji0j0n/ciudad_de_mexico_final_results.gpkg?rlkey=8k7904o43fe7d66v4vcgdeya2&dl=1'
}

primitive_cols = ['food_osm_count_1_2km', 'healthcare_osm_count_1_2km', 'educational_osm_count_1_2km', 'leisure_osm_count_1_2km', 'population_count_1_2km']
delta_hub = np.array([1, 0, 0, 1, 0], dtype=np.float32)

def download_and_filter(city, url):
    fname = f"{city}.gpkg"
    if not os.path.exists(fname): urllib.request.urlretrieve(url, fname)
    gdf = gpd.read_file(fname)
    return gdf[(gdf['population_count_1_2km'] > 0) & (gdf['network_density_1_2km'].notnull())].copy().reset_index(drop=True)

def compute_sumi(gdf):
    X = np.column_stack((gdf['network_density_1_2km'], gdf['intersection_density_ge3_1_2km'],
                         gdf['amenity_diversity_osm'], 1.0 / (gdf['average_link_length_1_2km'] + 1e-6)))
    X_norm = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0) + 1e-6) + 1e-6
    P = X_norm / X_norm.sum(axis=0)
    E = -(1.0 / np.log(X.shape[0])) * np.sum(P * np.log(P), axis=0)
    W = (1.0 - E) / (1.0 - E).sum()
    gdf['target_sumi'] = np.dot(X_norm, W)
    return gdf

def build_matrix(gdf, threshold=1200):
    coords = np.column_stack((gdf.geometry.centroid.x, gdf.geometry.centroid.y))
    tree = cKDTree(coords)
    A_mats = []
    for i in range(0, coords.shape[0], 10000):
        end = min(i + 10000, coords.shape[0])
        dist_mat = cKDTree(coords[i:end]).sparse_distance_matrix(tree, max_distance=threshold).tocoo()
        dist_mat.data = (1.0 - (dist_mat.data / threshold)).astype(np.float32)
        A_mats.append(dist_mat.tocsr())
        del dist_mat; gc.collect()
    A = sparse.vstack(A_mats)
    A.setdiag(1.0)
    return A.tocsc()

def train_gam(gdf):
    return LinearGAM(s(0, constraints='monotonic_inc') + s(1, constraints='monotonic_inc') +
                     s(2, constraints='monotonic_inc') + s(3, constraints='monotonic_inc') + s(4)).fit(gdf[primitive_cols].values, gdf['target_sumi'].values)

# ========================================================
# 2. INSTRUMENTED LAZY GREEDY ALGORITHM
# ========================================================
def lazy_greedy_diagnostics(A_csc, gam, budget, df, num_cands=2500):
    V_init = df[primitive_cols].values.astype(np.float32)
    pop = df['population_count_1_2km'].values.astype(np.float32)

    scores = df['target_sumi'].values
    cands = np.argsort(pop * (np.max(scores) - scores))[::-1][:num_cands]
    curr_V = V_init.copy()

    # --- DIAGNOSTIC COUNTERS ---
    eval_count = 0
    pop_count = 0
    stale_count = 0

    def get_gain(j, V_state):
        nonlocal eval_count
        eval_count += 1

        s_idx, e_idx = A_csc.indptr[j], A_csc.indptr[j+1]
        nbrs, decays = A_csc.indices[s_idx:e_idx], A_csc.data[s_idx:e_idx]
        if len(nbrs) == 0: return 0.0

        v_old = V_state[nbrs]
        v_new = v_old + np.outer(decays, delta_hub)
        p_aff = pop[nbrs]
        return np.sum((gam.predict(v_new) * p_aff) - (gam.predict(v_old) * p_aff))

    pq = []
    for j in cands:
        g = get_gain(j, curr_V)
        if g > 0: heapq.heappush(pq, (-g, j))

    st = time.time()

    for _ in range(budget):
        while pq:
            neg_g, j = heapq.heappop(pq)
            pop_count += 1

            act_g = get_gain(j, curr_V)

            if not pq or act_g >= -pq[0][0]:
                s_idx, e_idx = A_csc.indptr[j], A_csc.indptr[j+1]
                curr_V[A_csc.indices[s_idx:e_idx]] += np.outer(A_csc.data[s_idx:e_idx], delta_hub)
                break
            else:
                stale_count += 1
                heapq.heappush(pq, (-act_g, j))

    runtime = time.time() - st
    return eval_count, pop_count, stale_count, runtime

# ========================================================
# 3. EXECUTE DIAGNOSTICS ACROSS CITIES
# ========================================================
print("\n" + "="*50)
print("ALGORITHMIC DIAGNOSTICS (B=50, Cands=2500)")
print("="*50)

for city in ['zacatecas', 'monterrey', 'cdmx']:
    print(f"\nProcessing {city.upper()}...")
    gdf = download_and_filter(city, data_urls[city])
    gdf = compute_sumi(gdf)

    A = build_matrix(gdf, 1200)
    gam = train_gam(gdf)

    evals, pops, stales, time_taken = lazy_greedy_diagnostics(A, gam, 50, gdf)

    print(f"Runtime: {time_taken:.2f} seconds")
    print(f"Total Objective Evaluations: {evals:,}")
    print(f"Total Heap Pops: {pops:,}")
    print(f"Stale Bounds (Re-evaluations): {stales:,}")

    del gdf, A, gam
    gc.collect()


ALGORITHMIC DIAGNOSTICS (B=50, Cands=2500)

Processing ZACATECAS...
Runtime: 136.74 seconds
Total Objective Evaluations: 13,329
Total Heap Pops: 10,829
Stale Bounds (Re-evaluations): 10,779

Processing MONTERREY...
Runtime: 90.25 seconds
Total Objective Evaluations: 8,992
Total Heap Pops: 6,492
Stale Bounds (Re-evaluations): 6,442

Processing CDMX...
Runtime: 63.23 seconds
Total Objective Evaluations: 7,182
Total Heap Pops: 4,682
Stale Bounds (Re-evaluations): 4,632


#Radius vs. Stale Bounds

In [5]:
import gc
import time
import numpy as np

print("\n" + "="*50)
print("CAUSAL DIAGNOSTICS: RADIUS VS. STALE BOUNDS (Monterrey, B=50)")
print("="*50)

mty_gdf = download_and_filter('monterrey', data_urls['monterrey'])
if 'target_sumi' not in mty_gdf.columns:
    mty_gdf = compute_sumi(mty_gdf)

gam_mty = train_gam(mty_gdf)
radii_to_test = [600, 800, 1000, 1200, 1400, 1600]

print(f"{'Radius (m)':<15} | {'Stale Bounds':<15} | {'Time (s)':<10}")
print("-" * 45)

for r in radii_to_test:
    # Build a temporary matrix for the specific radius
    A_temp = build_matrix(mty_gdf, threshold=r)

    # Run the instrumented lazy greedy
    evals, pops, stales, time_taken = lazy_greedy_diagnostics(A_temp, gam_mty, budget=50, df=mty_gdf, num_cands=2500)

    # Print the live results
    print(f"{r:<15} | {stales:<15,} | {time_taken:<10.2f}")

    del A_temp
    gc.collect()

print("="*50)


CAUSAL DIAGNOSTICS: RADIUS VS. STALE BOUNDS (Monterrey, B=50)
Radius (m)      | Stale Bounds    | Time (s)  
---------------------------------------------
600             | 3,159           | 23.42     
800             | 4,344           | 43.15     
1000            | 5,449           | 63.43     
1200            | 6,442           | 88.78     
1400            | 7,154           | 121.46    
1600            | 7,608           | 148.25    


In [6]:
!pip install geopandas libpysal pygam tqdm contextily

# The Master Benchmarking Script

In [7]:
import os
import time
import gc
import urllib.request
import numpy as np
import geopandas as gpd
import scipy.sparse as sparse
from scipy.spatial import cKDTree
from pygam import LinearGAM, s
import heapq
from tqdm import tqdm

# ========================================================
# 1. SETUP & DATA INGESTION
# ========================================================
data_urls = {
    'zacatecas': 'https://www.dropbox.com/scl/fi/mtw6yanftigtyafoc493l/zacatecas_guadalupe_final_results.gpkg?rlkey=uibkahfz9i55z4m5idn3r84ou&dl=1',
    'monterrey': 'https://www.dropbox.com/scl/fi/hss7xxnfrl8fwk4prupvr/monterrey_final_results.gpkg?rlkey=6kri4ejs5juhpm9l699nkg5j1&dl=1',
    'cdmx': 'https://www.dropbox.com/scl/fi/wxps4aijd4mf2jmji0j0n/ciudad_de_mexico_final_results.gpkg?rlkey=8k7904o43fe7d66v4vcgdeya2&dl=1'
}

primitive_cols = ['food_osm_count_1_2km', 'healthcare_osm_count_1_2km', 'educational_osm_count_1_2km', 'leisure_osm_count_1_2km', 'population_count_1_2km']
delta_hub = np.array([1, 0, 0, 1, 0], dtype=np.float32)

def download_and_filter(city, url):
    fname = f"{city}.gpkg"
    if not os.path.exists(fname): urllib.request.urlretrieve(url, fname)
    gdf = gpd.read_file(fname)
    return gdf[(gdf['population_count_1_2km'] > 0) & (gdf['network_density_1_2km'].notnull())].copy().reset_index(drop=True)

def compute_sumi(gdf):
    X = np.column_stack((gdf['network_density_1_2km'], gdf['intersection_density_ge3_1_2km'],
                         gdf['amenity_diversity_osm'], 1.0 / (gdf['average_link_length_1_2km'] + 1e-6)))
    X_norm = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0) + 1e-6) + 1e-6
    P = X_norm / X_norm.sum(axis=0)
    E = -(1.0 / np.log(X.shape[0])) * np.sum(P * np.log(P), axis=0)
    W = (1.0 - E) / (1.0 - E).sum()
    gdf['target_sumi'] = np.dot(X_norm, W)
    return gdf

def build_matrix(gdf, threshold=1200):
    coords = np.column_stack((gdf.geometry.centroid.x, gdf.geometry.centroid.y))
    tree = cKDTree(coords)
    A_mats = []
    for i in range(0, coords.shape[0], 10000):
        end = min(i + 10000, coords.shape[0])
        dist_mat = cKDTree(coords[i:end]).sparse_distance_matrix(tree, max_distance=threshold).tocoo()
        dist_mat.data = (1.0 - (dist_mat.data / threshold)).astype(np.float32)
        A_mats.append(dist_mat.tocsr())
        del dist_mat; gc.collect()
    A = sparse.vstack(A_mats)
    A.setdiag(1.0)
    return A.tocsc()

def train_gam(gdf):
    return LinearGAM(s(0, constraints='monotonic_inc') + s(1, constraints='monotonic_inc') +
                     s(2, constraints='monotonic_inc') + s(3, constraints='monotonic_inc') + s(4)).fit(gdf[primitive_cols].values, gdf['target_sumi'].values)

# ========================================================
# 2. THE OPTIMIZER
# ========================================================
def lazy_greedy(A_csc, gam, budget, df, num_cands=2500):
    V_init = df[primitive_cols].values.astype(np.float32)
    pop = df['population_count_1_2km'].values.astype(np.float32)

    scores = df['target_sumi'].values
    cands = np.argsort(pop * (np.max(scores) - scores))[::-1][:num_cands]
    curr_V = V_init.copy()

    def get_gain(j, V_state):
        s_idx, e_idx = A_csc.indptr[j], A_csc.indptr[j+1]
        nbrs, decays = A_csc.indices[s_idx:e_idx], A_csc.data[s_idx:e_idx]
        if len(nbrs) == 0: return 0.0
        v_old = V_state[nbrs]
        v_new = v_old + np.outer(decays, delta_hub)
        p_aff = pop[nbrs]
        return np.sum((gam.predict(v_new) * p_aff) - (gam.predict(v_old) * p_aff))

    pq = []
    for j in cands:
        g = get_gain(j, curr_V)
        if g > 0: heapq.heappush(pq, (-g, j))

    obj = np.sum(gam.predict(curr_V) * pop)
    history = [obj]
    st = time.time()

    for _ in range(budget):
        while pq:
            neg_g, j = heapq.heappop(pq)
            act_g = get_gain(j, curr_V)
            if not pq or act_g >= -pq[0][0]:
                obj += act_g
                history.append(obj)
                s_idx, e_idx = A_csc.indptr[j], A_csc.indptr[j+1]
                curr_V[A_csc.indices[s_idx:e_idx]] += np.outer(A_csc.data[s_idx:e_idx], delta_hub)
                break
            else:
                heapq.heappush(pq, (-act_g, j))

    return obj, history, time.time() - st

# ========================================================
# 3. EXPERIMENT EXECUTION
# ========================================================
print("\n" + "="*50)
print("EXPERIMENT 1: APPLES-TO-APPLES SCALABILITY")
print("="*50)
scale_res = {}
for city in ['zacatecas', 'monterrey', 'cdmx']:
    print(f"Processing {city.upper()}...")
    gdf = compute_sumi(download_and_filter(city, data_urls[city]))
    A = build_matrix(gdf)
    _, _, runtime = lazy_greedy(A, train_gam(gdf), budget=50, df=gdf, num_cands=2500)
    scale_res[city] = {'nodes': len(gdf), 'time': runtime}
    del gdf, A; gc.collect()

print("\n" + "="*50)
print("EXPERIMENT 2: BUDGET SENSITIVITY (MONTERREY)")
print("="*50)
mty = compute_sumi(download_and_filter('monterrey', data_urls['monterrey']))
gam = train_gam(mty)
A_1200 = build_matrix(mty, 1200)

print("Running single continuous sequence to B=250...")
_, hist_B, _ = lazy_greedy(A_1200, gam, budget=250, df=mty, num_cands=10000)
budget_milestones = [10, 25, 50, 100, 250]
baseline_obj = hist_B[0]
budget_res = {b: (hist_B[b] - baseline_obj) for b in budget_milestones}

print("\n" + "="*50)
print("EXPERIMENT 3: DIFFUSION ROBUSTNESS (MONTERREY, B=50)")
print("="*50)
radii = [600, 800, 1000, 1200, 1500]
robust_res = {}

for r in radii:
    if r == 1200:
        obj, _, _ = lazy_greedy(A_1200, gam, budget=50, df=mty, num_cands=5000)
    else:
        print(f"Evaluating Radius: {r}m...")
        A_temp = build_matrix(mty, threshold=r)
        obj, _, _ = lazy_greedy(A_temp, gam, budget=50, df=mty, num_cands=5000)
        del A_temp; gc.collect()
    robust_res[r] = obj - baseline_obj

# ========================================================
# 4. FINAL LATEX TABLES OUPUT
# ========================================================
print("\n\n" + "#"*50)
print("FINAL RESULTS FOR LATEX MANUSCRIPT")
print("#"*50)

print("\n--- TABLE X: SCALABILITY ---")
print("City\t\tNodes\t\tRuntime (s)")
for c, d in scale_res.items(): print(f"{c.capitalize()}\t{d['nodes']:,}\t\t{d['time']:.2f}")

print("\n--- TABLE X: BUDGET SENSITIVITY ---")
print("Budget (B)\tMarginal Value Added")
for b, val in budget_res.items(): print(f"{b}\t\t{val:,.2f}")

print("\n--- TABLE X: DIFFUSION ROBUSTNESS (B=50) ---")
print("Radius (m)\tMarginal Value Added")
for r, val in robust_res.items(): print(f"{r}\t\t{val:,.2f}")


EXPERIMENT 1: APPLES-TO-APPLES SCALABILITY
Processing ZACATECAS...
Processing MONTERREY...
Processing CDMX...

EXPERIMENT 2: BUDGET SENSITIVITY (MONTERREY)
Running single continuous sequence to B=250...

EXPERIMENT 3: DIFFUSION ROBUSTNESS (MONTERREY, B=50)
Evaluating Radius: 600m...
Evaluating Radius: 800m...
Evaluating Radius: 1000m...
Evaluating Radius: 1500m...


##################################################
FINAL RESULTS FOR LATEX MANUSCRIPT
##################################################

--- TABLE X: SCALABILITY ---
City		Nodes		Runtime (s)
Zacatecas	42,782		145.34
Monterrey	303,782		90.67
Cdmx	539,179		64.63

--- TABLE X: BUDGET SENSITIVITY ---
Budget (B)	Marginal Value Added
10		13,885,293.00
25		30,031,166.53
50		51,459,649.95
100		83,885,709.75
250		143,230,112.00

--- TABLE X: DIFFUSION ROBUSTNESS (B=50) ---
Radius (m)	Marginal Value Added
600		16,901,792.27
800		26,820,547.93
1000		37,628,167.55
1200		49,041,007.30
1500		66,131,383.86
